<a href="https://colab.research.google.com/github/Zuhair0000/tensorflow_bootcamp/blob/main/08_intoduction_to_NLP_in_tensorflow.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!wget https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py

--2026-02-28 06:07:20--  https://raw.githubusercontent.com/mrdbourke/tensorflow-deep-learning/refs/heads/main/extras/helper_functions.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 10246 (10K) [text/plain]
Saving to: ‘helper_functions.py’

helper_functions.py 100%[===================>]  10.01K  --.-KB/s    in 0s      

2026-02-28 06:07:20 (105 MB/s) - ‘helper_functions.py’ saved [10246/10246]



In [2]:
from helper_functions import unzip_data, create_tensorboard_callback, plot_loss_curves, compare_historys

In [3]:
!wget https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip
unzip_data("nlp_getting_started.zip")

--2026-02-28 06:07:35--  https://storage.googleapis.com/ztm_tf_course/nlp_getting_started.zip
Resolving storage.googleapis.com (storage.googleapis.com)... 142.250.141.207, 142.250.101.207, 142.251.2.207, ...
Connecting to storage.googleapis.com (storage.googleapis.com)|142.250.141.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 607343 (593K) [application/zip]
Saving to: ‘nlp_getting_started.zip’

nlp_getting_started 100%[===================>] 593.11K  --.-KB/s    in 0.004s  

2026-02-28 06:07:35 (141 MB/s) - ‘nlp_getting_started.zip’ saved [607343/607343]



# Visualizing a text dataset

In [4]:
import pandas as pd
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

In [5]:
train_df.head()

,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [6]:
train_df_shuffled = train_df.sample(frac=1, random_state=42)
train_df_shuffled.head()

,id,keyword,location,text,target
2644,3796,destruction,NaN,So you have a new weapon that can cause un-ima...,1
2227,3185,deluge,NaN,The f$&amp;@ing things I do for #GISHWHES Just...,0
5448,7769,police,UK,DT @georgegalloway: RT @Galloway4Mayor: ÛÏThe...,1
132,191,aftershock,NaN,Aftershock back to school kick off was great. ...,0
6845,9810,trauma,"Montgomery County, MD",in response to trauma Children of Addicts deve...,0


In [7]:
test_df.head()

,id,keyword,location,text
0,0,NaN,NaN,Just happened a terrible car crash
1,2,NaN,NaN,"Heard about #earthquake is different cities, s..."
2,3,NaN,NaN,"there is a forest fire at spot pond, geese are..."
3,9,NaN,NaN,Apocalypse lighting. #Spokane #wildfires
4,11,NaN,NaN,Typhoon Soudelor kills 28 in China and Taiwan


In [8]:
train_df_shuffled.target.value_counts()

,count
target,
0,4342
1,3271


In [9]:
len(train_df_shuffled)

7613

In [10]:
len(test_df)

3263

In [11]:
import random
random_index = random.randint(0, len(train_df)-5)
for row in train_df_shuffled[['text', 'target']][random_index: random_index+5].itertuples():
  _, text, target = row
  print(f"Target: {target}", "(real disaster)" if target > 0 else "(not real disaster)")
  print(f"Text:\n{text}\n")
  print("---\n")

Target: 0 (not real disaster)
Text:
@UABStephenLong @courtlizcamp Total tweet fail! You are so beautiful inside and out Blaze On!

---

Target: 1 (real disaster)
Text:
Road closures remain in effect due to hazard trees falling tree torching and uphill runs of the fire. Forest Service Road #1 remains close

---

Target: 0 (not real disaster)
Text:
#socialmedia news - New Facebook Page Features Seek to Help Personalize the Customer Experience http://t.co/nbizaTlsmV

---

Target: 1 (real disaster)
Text:
News786-UK Islamist Cleric Anjem Choudary Charged Under Terrorism Act: http://t.co/u7bBeNXWYK

---

Target: 0 (not real disaster)
Text:
http://t.co/kG5pLkeDhr WRAPUP 2-U.S. cable TV companies' shares crushed after Disney disappoints http://t.co/QeIhvn3DNQ

---



# Split Data

In [12]:
from sklearn.model_selection import train_test_split
train_sentences, val_sentences, train_labels, val_labels = train_test_split(train_df_shuffled['text'].to_numpy(),
                                                                            train_df_shuffled['target'].to_numpy(),
                                                                            test_size=0.1,
                                                                            random_state=42)


In [13]:
len(train_sentences), len(train_labels), len(val_sentences), len(val_labels)

(6851, 6851, 762, 762)

# Converting text to numbers

In [14]:
train_sentences[:5]

array(['@mogacola @zamtriossu i screamed after hitting tweet',
       'Imagine getting flattened by Kurt Zouma',
       '@Gurmeetramrahim #MSGDoing111WelfareWorks Green S welfare force ke appx 65000 members har time disaster victim ki help ke liye tyar hai....',
       "@shakjn @C7 @Magnums im shaking in fear he's gonna hack the planet",
       'Somehow find you and I collide http://t.co/Ee8RpOahPk'],
      dtype=object)

In [15]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

text_vectorizer = TextVectorization(max_tokens=None,
                                    standardize='lower_and_strip_punctuation',
                                    split="whitespace",
                                    ngrams=None,
                                    output_mode='int',
                                    output_sequence_length=None,
                                    pad_to_max_tokens=False
                                   )

In [16]:
round(sum([len(i.split()) for i in train_sentences])/len(train_sentences))

15

In [17]:
max_vocab_length = 10000
max_length = 15

text_vectorizer = TextVectorization(max_tokens=max_vocab_length,
                                    output_mode = 'int',
                                    output_sequence_length=max_length)

In [18]:
text_vectorizer.adapt(train_sentences)

In [19]:
sample_sentence = "There's a flood in my street!"
text_vectorizer([sample_sentence])

<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[264,   3, 232,   4,  13, 698,   0,   0,   0,   0,   0,   0,   0,
          0,   0]])>

In [20]:
random_sentence = random.choice(train_sentences)
print(f" Original text:\n {random_sentence} \
        \n\nVectorized version:")
text_vectorizer([random_sentence])

 Original text:
 Cruise's 'M:I 5' emergency plan: Awesome fail http://t.co/H3dCh6Fyaw         

Vectorized version:


<tf.Tensor: shape=(1, 15), dtype=int64, numpy=
array([[   1,    1,  180,   73,  241, 1042, 2118,    1,    0,    0,    0,
           0,    0,    0,    0]])>

In [21]:
words_in_vocab = text_vectorizer.get_vocabulary()
top_5_words = words_in_vocab[:5]
bottom_5_words = words_in_vocab[-5:]

print(f"Number of words in vocab: {len(words_in_vocab)}")
print(f"5 most common words: {top_5_words}")
print(f"5 least common words: {bottom_5_words}")

Number of words in vocab: 10000
5 most common words: ['', '[UNK]', np.str_('the'), np.str_('a'), np.str_('in')]
5 least common words: [np.str_('pages'), np.str_('paeds'), np.str_('pads'), np.str_('padres'), np.str_('paddytomlinson1')]


# Creating an embedding layer

In [22]:
embedding = tf.keras.layers.Embedding(input_dim=max_vocab_length,
                                      output_dim=128,
                                      input_length=max_length
                                      )

embedding

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


<Embedding name=embedding, built=False>

In [23]:
random_sentences = random.choice(train_sentences)
print(f"Original text:\n {random_sentence}\
        \n\nEmbedded version:")
sample_embed = embedding(text_vectorizer([random_sentence]))
sample_embed

Original text:
 Cruise's 'M:I 5' emergency plan: Awesome fail http://t.co/H3dCh6Fyaw        

Embedded version:


<tf.Tensor: shape=(1, 15, 128), dtype=float32, numpy=
array([[[ 0.01956787,  0.03891846,  0.01730852, ..., -0.00019959,
          0.03604538,  0.01676032],
        [ 0.01956787,  0.03891846,  0.01730852, ..., -0.00019959,
          0.03604538,  0.01676032],
        [-0.03855975,  0.00079615,  0.02838191, ..., -0.01867204,
         -0.00065476, -0.03503612],
        ...,
        [-0.03600423, -0.01037646,  0.03988585, ..., -0.03063521,
         -0.01029425, -0.03280435],
        [-0.03600423, -0.01037646,  0.03988585, ..., -0.03063521,
         -0.01029425, -0.03280435],
        [-0.03600423, -0.01037646,  0.03988585, ..., -0.03063521,
         -0.01029425, -0.03280435]]], dtype=float32)>

## Modelling a text dataset

### Model 0

In [24]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

model_0 = Pipeline([
    ('tfidf', TfidfVectorizer()),
    ('model', MultinomialNB())
])

model_0.fit(train_sentences, train_labels)

Pipeline(steps=[('tfidf', TfidfVectorizer()), ('model', MultinomialNB())])

In [25]:
baseline_score = model_0.score(val_sentences, val_labels)
print(f"Baseline model acieves an accuracy of: {baseline_score*100:.2f}")

Baseline model acieves an accuracy of: 79.27


In [26]:
baseline_prediction = model_0.predict(val_sentences)

In [27]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def evaluate(y_pred, y_true):
  model_precision, model_recall, model_f1, _ = precision_recall_fscore_support(y_true, y_pred, average='weighted')
  return{
      "Accuracy Score": accuracy_score(y_true, y_pred),
      "Model Precision": model_precision,
      "Recall": model_recall,
      "F1": model_f1
  }

In [28]:
baseline_results = evaluate(val_labels, baseline_prediction)
baseline_results

{'Accuracy Score': 0.7926509186351706,
 'Model Precision': 0.8336022277575122,
 'Recall': 0.7926509186351706,
 'F1': 0.7990828614653861}

### Model 1

In [29]:
from helper_functions import create_tensorboard_callback

SAVE_DIR = 'model_logs'

In [30]:
inputs = tf.keras.layers.Input(shape=(1,), dtype=tf.string)
x = text_vectorizer(inputs)
x = embedding(x)
x = tf.keras.layers.GlobalAveragePooling1D()(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)


model_1 = tf.keras.Model(inputs, outputs, name='model_1_dense')

In [31]:
model_1.compile(loss='binary_crossentropy',
                optimizer='Adam',
                metrics=['accuracy'])

In [32]:
history_1 = model_1.fit(x = train_sentences,
                        y=train_labels,
                        epochs=5,
                        validation_data = (val_sentences, val_labels),
                        callbacks=[create_tensorboard_callback(dir_name='SAVE_DIR',
                                                               experiment_name='model_1_dense')])

Saving TensorBoard log files to: SAVE_DIR/model_1_dense/20260228-060742
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.6540 - loss: 0.6472 - val_accuracy: 0.7651 - val_loss: 0.5323
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 6ms/step - accuracy: 0.8211 - loss: 0.4536 - val_accuracy: 0.7835 - val_loss: 0.4694
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8583 - loss: 0.3490 - val_accuracy: 0.7848 - val_loss: 0.4575
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.8934 - loss: 0.2881 - val_accuracy: 0.7887 - val_loss: 0.4674
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9136 - loss: 0.2372 - val_accuracy: 0.7861 - val_loss: 0.4830


In [33]:
model_1.evaluate(val_sentences, val_labels)

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.7718 - loss: 0.5190


[0.48300066590309143, 0.7860892415046692]

In [34]:
model_1_pred_probs = model_1.predict(val_sentences)
model_1_pred_probs.shape

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


(762, 1)

In [35]:
model_1_preds = tf.squeeze(tf.round(model_1_pred_probs))
model_1_preds[:20]

<tf.Tensor: shape=(20,), dtype=float32, numpy=
array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0., 0., 1., 0., 0., 0., 0., 0.,
       0., 0., 0.], dtype=float32)>

In [36]:
model_1_results = evaluate(val_labels,
                           model_1_preds)
model_1_results, baseline_results

({'Accuracy Score': 0.7860892388451444,
  'Model Precision': 0.8092854819309505,
  'Recall': 0.7860892388451444,
  'F1': 0.7901232180827413},
 {'Accuracy Score': 0.7926509186351706,
  'Model Precision': 0.8336022277575122,
  'Recall': 0.7926509186351706,
  'F1': 0.7990828614653861})

# Visualizing Embeddings

In [37]:
words_in_vocab = text_vectorizer.get_vocabulary()
len(words_in_vocab), words_in_vocab[:10]

(10000,
 ['',
  '[UNK]',
  np.str_('the'),
  np.str_('a'),
  np.str_('in'),
  np.str_('to'),
  np.str_('of'),
  np.str_('and'),
  np.str_('i'),
  np.str_('is')])

In [38]:
model_1.summary()

Model: "model_1_dense"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 1)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ text_vectorization_1            │ (None, 15)             │             0 │
│ (TextVectorization)             │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 15, 128)        │     1,280,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,840,389 (14.65 MB)

 Trainable params: 1,280,129 (4.88 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 2,560,260 (9.77 MB)

In [39]:
embed_weights = model_1.get_layer('embedding').get_weights()[0]
print(embed_weights.shape)

(10000, 128)


In [40]:
import io
out_v = io.open("vectors.tsv", 'w', encoding='utf-8')
out_m = io.open("metadata.tsv", 'w', encoding='utf-8')

for index, word in enumerate(words_in_vocab):
  if index == 0:
    continue
  vec = embed_weights[index]
  out_v.write('\t'.join([str(x) for x in vec]) + '\n')
  out_m.write(word + '\n')
out_v.close()
out_m.close()

In [41]:
try:
  from google.colab import files
  files.download('vectors.tsv')
  files.download('metadata.tsv')
except:
  pass

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# RNN

## LSTM

In [42]:
inputs = tf.keras.layers.Input(shape=(1,), dtype='string')
x = text_vectorizer(inputs)
x = embedding(x)
x = tf.keras.layers.LSTM(64, return_sequences=True)(x)
x = tf.keras.layers.LSTM(64)(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model_2 = tf.keras.Model(inputs, outputs, name='model_2_LSTM')

In [43]:
model_2.compile(loss='binary_crossentropy',
                optimizer='Adam',
                metrics=['accuracy'])

In [44]:
history_2 = model_2.fit(train_sentences,
                        train_labels,
                        epochs=5,
                        validation_data=(val_sentences, val_labels),
                        callbacks=[create_tensorboard_callback(dir_name=SAVE_DIR,
                                                               experiment_name='model_2_LSTM')])

Saving TensorBoard log files to: model_logs/model_2_LSTM/20260228-060800
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 11ms/step - accuracy: 0.9072 - loss: 0.2903 - val_accuracy: 0.7756 - val_loss: 0.6024
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9479 - loss: 0.1481 - val_accuracy: 0.7835 - val_loss: 0.6192
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9478 - loss: 0.1346 - val_accuracy: 0.7861 - val_loss: 0.6559
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9625 - loss: 0.1067 - val_accuracy: 0.7743 - val_loss: 0.7670
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.9704 - loss: 0.0805 - val_accuracy: 0.7690 - val_loss: 1.0447


In [45]:
model_2_pred_probs = model_2.predict(val_sentences)
model_2_pred_probs[:10]

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step


array([[1.4434551e-02],
       [8.2584840e-01],
       [9.9994767e-01],
       [1.3804156e-01],
       [1.2694567e-04],
       [9.9987400e-01],
       [9.7650027e-01],
       [9.9997497e-01],
       [9.9995577e-01],
       [4.7326067e-01]], dtype=float32)

In [46]:
model_2_preds = tf.squeeze(tf.round(model_2_pred_probs))
model_2_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0.], dtype=float32)>

In [47]:
model_2_results = evaluate(val_labels,
                           model_2_preds)
model_2_results

{'Accuracy Score': 0.7690288713910761,
 'Model Precision': 0.770763786960413,
 'Recall': 0.7690288713910761,
 'F1': 0.7695568789892343}

In [48]:
baseline_results

{'Accuracy Score': 0.7926509186351706,
 'Model Precision': 0.8336022277575122,
 'Recall': 0.7926509186351706,
 'F1': 0.7990828614653861}

## GRU

In [49]:
inputs = tf.keras.layers.Input(shape=(1,), dtype='string')
x = text_vectorizer(inputs)
x = embedding(x)
x = tf.keras.layers.GRU(64, return_sequences=True)(x)
x = tf.keras.layers.LSTM(64, return_sequences=True)(x)
x = tf.keras.layers.GRU(64)(x)
x = tf.keras.layers.Dense(64, activation='relu')(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

model_3 = tf.keras.Model(inputs, outputs, name='model_3_GRU')

In [50]:
model_3.compile(loss='binary_crossentropy',
                optimizer='Adam',
                metrics=['accuracy'])

In [51]:
history_3 = model_3.fit(train_sentences,
                        train_labels,
                        epochs=5,
                        validation_data=(val_sentences, val_labels),
                        callbacks=[create_tensorboard_callback(dir_name=SAVE_DIR,
                                                               experiment_name='model_3_GRU')])

Saving TensorBoard log files to: model_logs/model_3_GRU/20260228-060814
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 12ms/step - accuracy: 0.9154 - loss: 0.2230 - val_accuracy: 0.7795 - val_loss: 0.8345
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9640 - loss: 0.0854 - val_accuracy: 0.7585 - val_loss: 1.0178
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 12ms/step - accuracy: 0.9753 - loss: 0.0644 - val_accuracy: 0.7677 - val_loss: 1.2438
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9728 - loss: 0.0554 - val_accuracy: 0.7730 - val_loss: 1.3096
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9691 - loss: 0.0696 - val_accuracy: 0.7703 - val_loss: 1.4719


In [52]:
model_3_pred_probs = model_3.predict(val_sentences)
model_3_pred_probs[:10]

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step


array([[4.5254809e-04],
       [6.8300968e-01],
       [9.9993932e-01],
       [3.3110064e-03],
       [6.1258776e-05],
       [9.9954587e-01],
       [9.6166551e-01],
       [9.9994671e-01],
       [9.9994075e-01],
       [3.2111242e-01]], dtype=float32)

In [53]:
model_3_preds = tf.squeeze(tf.round(model_3_pred_probs))
model_3_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 0.], dtype=float32)>

In [54]:
model_3_results = evaluate(val_labels,
                           model_3_preds)
model_3_results

{'Accuracy Score': 0.7703412073490814,
 'Model Precision': 0.7855255989677952,
 'Recall': 0.7703412073490814,
 'F1': 0.773338865119529}

# Bidirectional LSTM

In [56]:
inputs = tf.keras.layers.Input(shape=(1,), dtype='string')
x = text_vectorizer(inputs)
x = embedding(x)
x = tf.keras.layers.Bidirectional(tf.keras.layers.LSTM(64, return_sequences=True))(x)
x = tf.keras.layers.Bidirectional(tf.keras.layers.GRU(64))(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model_4 = tf.keras.Model(inputs, outputs)

In [57]:
model_4.compile(loss='binary_crossentropy',
                optimizer='Adam',
                metrics=['accuracy'])

In [58]:
history_4 = model_4.fit(train_sentences,
                        train_labels,
                        epochs=5,
                        validation_data=(val_sentences, val_labels),
                        callbacks=[create_tensorboard_callback(dir_name=SAVE_DIR,
                                                               experiment_name='model_4_Bidirectional')])

Saving TensorBoard log files to: model_logs/model_4_Bidirectional/20260228-061448
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 12s 21ms/step - accuracy: 0.9512 - loss: 0.1764 - val_accuracy: 0.7782 - val_loss: 0.9708
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 25ms/step - accuracy: 0.9771 - loss: 0.0519 - val_accuracy: 0.7703 - val_loss: 0.9305
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 6s 27ms/step - accuracy: 0.9770 - loss: 0.0507 - val_accuracy: 0.7782 - val_loss: 1.2463
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 13s 37ms/step - accuracy: 0.9746 - loss: 0.0483 - val_accuracy: 0.7756 - val_loss: 1.4510
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 3s 15ms/step - accuracy: 0.9783 - loss: 0.0408 - val_accuracy: 0.7743 - val_loss: 1.5394


In [59]:
model_4_pred_probs = model_4.predict(val_sentences)
model_4_pred_probs[:10]

24/24 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step


array([[2.1483309e-03],
       [5.1157540e-01],
       [9.9996042e-01],
       [2.3666725e-01],
       [6.6008110e-06],
       [9.9990714e-01],
       [4.6270356e-01],
       [9.9999130e-01],
       [9.9997640e-01],
       [4.2585200e-01]], dtype=float32)

In [60]:
model_4_preds = tf.squeeze(tf.round(model_4_pred_probs))
model_4_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 0., 1., 1., 0.], dtype=float32)>

In [61]:
model_4_results = evaluate(val_labels,
                           model_4_preds)
model_4_results

{'Accuracy Score': 0.7742782152230971,
 'Model Precision': 0.7880569776354167,
 'Recall': 0.7742782152230971,
 'F1': 0.777003984513248}

# Convolutional Nueral Netowrks for Text

In [63]:
embedding_test = embedding(text_vectorizer(['this is a test sentence']))
conv_1d = tf.keras.layers.Conv1D(filters=32,
                                 kernel_size=5, # It means it lokks at 5 words at a time
                                 activation='relu',
                                 padding='valid', #default = 'valid', the output is smaller the the input shape, 'same' means output is
                                 )
conv_1d_output = conv_1d(embedding_test) # pass test embedding through conv1D layer
max_pool = tf.keras.layers.GlobalMaxPool1D()
max_pool_output = max_pool(conv_1d_output)

embedding_test.shape, conv_1d_output.shape, max_pool_output.shape

(TensorShape([1, 15, 128]), TensorShape([1, 11, 32]), TensorShape([1, 32]))

In [64]:
inputs = tf.keras.layers.Input(shape=(1,), dtype=tf.string)
x = text_vectorizer(inputs)
x = embedding(x)
x = tf.keras.layers.Conv1D(64, 5, activation='relu', padding='valid')(x)
x = tf.keras.layers.GlobalMaxPool1D()(x)
outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)
model_5 = tf.keras.Model(inputs, outputs)

In [65]:
model_5.compile(loss='binary_crossentropy',
                optimizer='Adam',
                metrics=['accuracy'])

In [66]:
history_5 = model_5.fit(train_sentences,
                        train_labels,
                        epochs=5,
                        validation_data=(val_sentences, val_labels),
                        callbacks=[create_tensorboard_callback(dir_name=SAVE_DIR,
                                                               experiment_name='Conv1D')])

Saving TensorBoard log files to: model_logs/Conv1D/20260228-071435
Epoch 1/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9567 - loss: 0.1741 - val_accuracy: 0.7730 - val_loss: 0.8783
Epoch 2/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9776 - loss: 0.0641 - val_accuracy: 0.7625 - val_loss: 0.9833
Epoch 3/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9770 - loss: 0.0591 - val_accuracy: 0.7612 - val_loss: 1.1025
Epoch 4/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 4s 19ms/step - accuracy: 0.9778 - loss: 0.0512 - val_accuracy: 0.7625 - val_loss: 1.1553
Epoch 5/5
215/215 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - accuracy: 0.9809 - loss: 0.0450 - val_accuracy: 0.7598 - val_loss: 1.2118


In [67]:
model_5_pred_probs = model_5.predict(val_sentences)
model_5_pred_probs[:10]

24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step


array([[1.2483829e-01],
       [9.5648938e-01],
       [9.9993837e-01],
       [4.8364013e-02],
       [6.6988252e-07],
       [9.9799585e-01],
       [9.9432558e-01],
       [9.9993622e-01],
       [9.9999976e-01],
       [7.7853900e-01]], dtype=float32)

In [68]:
model_5_preds = tf.squeeze(tf.round(model_5_pred_probs))
model_5_preds[:10]

<tf.Tensor: shape=(10,), dtype=float32, numpy=array([0., 1., 1., 0., 0., 1., 1., 1., 1., 1.], dtype=float32)>

In [69]:
model_5_results = evaluate(val_labels,
                           model_5_preds)
model_5_results

{'Accuracy Score': 0.7598425196850394,
 'Model Precision': 0.7686248192614106,
 'Recall': 0.7598425196850394,
 'F1': 0.7618215832145521}